In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)


In [0]:
df.write.format("delta").mode("overWrite").save("/Volumes/volume1/default/delta_work")

In [0]:
df1=spark.read.format("delta").load("/Volumes/volume1/default/delta_work")
df1.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
#new data
new_data=[(5, "C005", "Camera", 30000)]
columns = ["id", "customer_id", "product", "amount"]
new_df=spark.createDataFrame(new_data,columns)


In [0]:
new_df.write.format("delta").mode("append").save("/Volumes/volume1/default/delta_work")

In [0]:
existing_df=spark.read.load("/Volumes/volume1/default/delta_work")
existing_df.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


In [0]:
### TASK 3: Update Data

 #Update:

#id = 2 → amount = 18000
from delta.tables import DeltaTable
deltatable=DeltaTable.forPath(spark,"/Volumes/volume1/default/delta_work")
deltatable.update("id=2",
                  set={"amount":lit(8000)})
deltatable.toDF().display()

id,customer_id,product,amount
1,C001,Laptop,50000
3,C003,Tablet,20000
4,C004,Laptop,55000
2,C002,Mobile,8000
5,C005,Camera,30000


In [0]:
#TASK 4: Delete Data
deltatable.delete('id=1')
deltatable.toDF().display()

id,customer_id,product,amount
2,C002,Mobile,8000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


In [0]:
#TASK 5: MERGE (Incremental Load)

 #New incoming data:

#updates = [
  #  (3, "C003", "Tablet", 22000),  # update
   # (6, "C006", "Watch", 8000)     # insert
#]

target_df=spark.read.load("/Volumes/volume1/default/delta_work")
updates=[
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]
columns = ["id", "customer_id", "product", "amount"]
update_df=spark.createDataFrame(updates,columns)
update_df.display()

id,customer_id,product,amount
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:
deltatable.alias("t").merge(update_df.alias("u"),"t.id=u.id").whenMatchedUpdate(set={"amount":"u.amount"}
    ).whenNotMatchedInsert(values={
    'id':'u.id','customer_id':'u.customer_id','product':'u.product','amount':'u.amount'
}).execute()



deltatable.toDF().display()


id,customer_id,product,amount
2,C002,Mobile,8000
4,C004,Laptop,55000
5,C005,Camera,30000
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:
#TASK 6: Schema Evolution

# Add new column:

#country

data=[(7,'C007','Headphones',10000,'USA'),
      (8,'C008','Keyboard',20000,'Canada')]
new_df=spark.createDataFrame(data,['id','customer_id','product','amount','country'])
new_df.display()


id,customer_id,product,amount,country
7,C007,Headphones,10000,USA
8,C008,Keyboard,20000,Canada


In [0]:
new_df.write.format("delta").mode("append").option("mergeSchema","true").save("/Volumes/volume1/default/delta_work")
df_new=spark.read.load("/Volumes/volume1/default/delta_work")
df_new.display()

id,customer_id,product,amount,country
7,C007,Headphones,10000,USA
8,C008,Keyboard,20000,Canada
7,C007,Headphones,10000,USA
8,C008,Keyboard,20000,Canada
2,C002,Mobile,8000,null
4,C004,Laptop,55000,null
5,C005,Camera,30000,null
3,C003,Tablet,22000,null
6,C006,Watch,8000,null


In [0]:
#TASK 7: Time Travel

#Previous version
#Restore old data
deltatable.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
30,2026-04-14T14:11:35.000Z,78380338472481,pandunallamelli@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2904921969146459),5099b652-9200-46c5-bf41-e347e7f99baa,0414-123526-q8qwygbk-v2n,29,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1500)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
29,2026-04-14T14:11:01.000Z,78380338472481,pandunallamelli@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2904921969146459),c3c3073a-2b30-4273-a47b-c6d9341e5f5e,0414-123526-q8qwygbk-v2n,28,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1500)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
28,2026-04-14T14:02:11.000Z,78380338472481,pandunallamelli@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2904921969146459),857b3afd-101c-4291-8aa1-bb48ea5b6ce0,0414-123526-q8qwygbk-v2n,27,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3852, p25FileSize -> 1377, numDeletionVectorsRemoved -> 1, minFileSize -> 1377, numAddedFiles -> 1, maxFileSize -> 1377, p75FileSize -> 1377, p50FileSize -> 1377, numAddedBytes -> 1377)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
27,2026-04-14T14:02:07.000Z,78380338472481,pandunallamelli@gmail.com,MERGE,"Map(predicate -> [""(id#17010L = id#19154L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2904921969146459),857b3afd-101c-4291-8aa1-bb48ea5b6ce0,0414-123526-q8qwygbk-v2n,26,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3516, materializeSourceTimeMs -> 112, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1315, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1936)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
26,2026-04-14T14:01:54.000Z,78380338472481,pandunallamelli@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2904921969146459),bf9be2e4-b0b6-4634-9eba-5ea7e0d3ce0c,0414-123526-q8qwygbk-v2n,25,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3852, p25FileSize -> 1377, numDeletionVectorsRemoved -> 1, minFileSize -> 1377, numAddedFiles -> 1, maxFileSize -> 1377, p75FileSize -> 1377, p50FileSize -> 1377, numAddedBytes -> 1377)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
25,2026-04-14T14:01:52.000Z,78380338472481,pandunallamelli@gmail.com,MERGE,"Map(predicate -> [""(id#17010L = id#18787L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2904921969146459),bf9be2e4-b0b6-4634-9eba-5ea7e0d3ce0c,0414-123526-q8qwygbk-v2n,24,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4248, materializeSourceTimeMs -> 121, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVec

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/volume1/default/delta_work")
df_old.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
deltatable.restoreToVersion(0)


DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [0]:
deltatable.toDF().display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000
